# Model Evaluation Notebook
**Arkesh Das**

In [2]:
# Import initial libraries
import pandas as pd
import numpy as np
import boto3
from io import StringIO
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
s3 = boto3.client('s3')
response = s3.list_buckets()
bucket_count = len(response['Buckets'])

print(f"Connected to S3")
print(f"Found {bucket_count} S3 buckets")

# Print your buckets (to identify your modeling data bucket)
for bucket in response['Buckets']:
    print(f"  - {bucket['Name']}")

Connected to S3
Found 1 S3 buckets
  - cmse492-dasarkes-nyc311-471112527784-us-east-1-an


In [5]:
bucket_name = 'cmse492-dasarkes-nyc311-471112527784-us-east-1-an'
file_name = 'modeling/q2_modeling_data.csv' 
obj = s3.get_object(Bucket=bucket_name, Key=file_name)
data = obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(data))

## Cleaning the data

In [6]:
# First few rows
print("First few rows:")
print(df.head())

# Data types and missing values
print("\n" + "="*50)
print("Data types and missing values:")
print(df.info())

First few rows:
  agency   borough             problem  incident_zip  day_of_week  \
0   DCWP     BRONX  Consumer Complaint       10455.0            5   
1   DCWP  BROOKLYN  Consumer Complaint       11207.0            5   
2   DCWP  BROOKLYN  Consumer Complaint       11234.0            5   
3   DCWP     BRONX  Consumer Complaint       10454.0            5   
4   DCWP    QUEENS  Consumer Complaint       11423.0            5   

   hour_of_day  same_day_complaint_volume  days_to_close  
0           13                         19             33  
1           11                         19              4  
2            3                         19             34  
3           11                         19             33  
4           15                         19             33  

Data types and missing values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173851 entries, 0 to 173850
Data columns (total 8 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------     

In [7]:
print(f"Shape: {df.shape}")

Shape: (173851, 8)


In [8]:
df.isna().sum().sort_values(ascending=False)

incident_zip                 1765
agency                          0
borough                         0
problem                         0
day_of_week                     0
hour_of_day                     0
same_day_complaint_volume       0
days_to_close                   0
dtype: int64

In [9]:
df = df.dropna().copy()

In [10]:
print(f"Shape: {df.shape}")

Shape: (172086, 8)


## Learning some more about the data

In [11]:
# Balance check based on the example query

# Check target balance
print("Target distribution (volume_quartile):")
print(df['days_to_close'].value_counts().sort_index())

# As percentage
print("\nTarget distribution (%)")
print((df['days_to_close'].value_counts() / len(df) * 100).sort_index().round(2))

Target distribution (volume_quartile):
days_to_close
0     106992
1      17454
2      12225
3       8085
4       5758
5       4518
6       3094
7       1909
8       1396
9       1229
10       819
11       722
12       677
13       606
14       551
15       490
16       444
17       430
18       375
19       391
20       321
21       281
22       271
23       264
24       214
25       224
26       230
27       186
28       167
29       170
30       225
31       153
32       161
33       137
34        97
35       106
36        98
37        76
38        78
39        69
40        85
41        70
42        53
43        37
44        34
45        38
46        29
47        16
48        19
49        12
Name: count, dtype: int64

Target distribution (%)
days_to_close
0     62.17
1     10.14
2      7.10
3      4.70
4      3.35
5      2.63
6      1.80
7      1.11
8      0.81
9      0.71
10     0.48
11     0.42
12     0.39
13     0.35
14     0.32
15     0.28
16     0.26
17     0.25
18     0.22
19  

In [12]:
# Summary statistics
print("Summary statistics:")
print(df.describe())

Summary statistics:
        incident_zip    day_of_week    hour_of_day  same_day_complaint_volume  \
count  172086.000000  172086.000000  172086.000000              172086.000000   
mean    10838.974629       3.899376      13.012837                 440.569163   
std       528.743746       1.991713       6.035696                 649.023290   
min     10000.000000       1.000000       0.000000                   1.000000   
25%     10452.000000       2.000000       9.000000                  52.000000   
50%     11205.000000       4.000000      13.000000                 262.000000   
75%     11235.000000       6.000000      18.000000                 560.000000   
max     11697.000000       7.000000      23.000000                3759.000000   

       days_to_close  
count  172086.000000  
mean        2.059871  
std         5.130181  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         2.000000  
max        49.000000  


In [13]:
# Balance check based on the example query, the target variable is `volume_quartile` -- make sure you're checking the right target variable for your modeling table!

# Check target balance
print("Target distribution (volume_quartile):")
print(df['days_to_close'].value_counts().sort_index())

# As percentage
print("\nTarget distribution (%)")
print((df['days_to_close'].value_counts() / len(df) * 100).sort_index().round(2))

Target distribution (volume_quartile):
days_to_close
0     106992
1      17454
2      12225
3       8085
4       5758
5       4518
6       3094
7       1909
8       1396
9       1229
10       819
11       722
12       677
13       606
14       551
15       490
16       444
17       430
18       375
19       391
20       321
21       281
22       271
23       264
24       214
25       224
26       230
27       186
28       167
29       170
30       225
31       153
32       161
33       137
34        97
35       106
36        98
37        76
38        78
39        69
40        85
41        70
42        53
43        37
44        34
45        38
46        29
47        16
48        19
49        12
Name: count, dtype: int64

Target distribution (%)
days_to_close
0     62.17
1     10.14
2      7.10
3      4.70
4      3.35
5      2.63
6      1.80
7      1.11
8      0.81
9      0.71
10     0.48
11     0.42
12     0.39
13     0.35
14     0.32
15     0.28
16     0.26
17     0.25
18     0.22
19  

In [14]:
print(f"Shape: {df.shape}")

Shape: (172086, 8)


## Setting the Predictor and Target variables

In [15]:
# Target
y = df['days_to_close']

# Features
X = df.drop(columns=['days_to_close'])

### Simple 80/20 split

In [16]:
from sklearn.model_selection import train_test_split

y = df['days_to_close']
X = df.drop(columns=['days_to_close'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=492
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")

Train size: 137668
Test size:  34418


In [17]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols),
    ]
)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
feature_names = ohe_names + num_cols

In [18]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_enc, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [19]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = lr.predict(X_test_enc)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2:  {r2:.3f}")
print(f"MAE: {mae:.3f}")

R2:  0.383
MAE: 1.846


In [20]:
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': lr.coef_
}).sort_values(by='coefficient', key=abs, ascending=False)

coef_df.head(15)

,feature,coefficient
117,problem_Radioactive Material,33.066457
66,problem_For Hire Vehicle Report,21.334049
90,problem_Lost Property,-15.250190
140,problem_Taxi Compliment,14.578381
83,problem_Institution Disposal Complaint,14.481384
70,problem_Green Taxi Complaint,-14.024622
37,problem_Cannabis Retailer,13.506998
10,agency_OOS,13.506998
63,problem_Food Establishment,12.626188
129,problem_Smoking or Vaping,9.848185


### Stratified K Fold Cross Validation

Because the data is so skewed, I think that a smarter thing to do instead of an 80/20 train test split is to do a stratified split. I am following the documentation from this tutorial: [https://www.geeksforgeeks.org/machine-learning/stratified-k-fold-cross-validation/](https://www.geeksforgeeks.org/machine-learning/stratified-k-fold-cross-validation/)

I'm making bins for the sake of making the strata, but I will still run basic linear regression on the `days_till_close`.

In [21]:
def stratify_bins(y):
    return pd.cut(
        y,
        bins=[-1, 0, 2, 5, 10, 50],
        labels=[0, 1, 2, 3, 4]
    )

y_strat = stratify_bins(df['days_to_close'])

In [22]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
r2_scores = []

In [23]:
for train_idx, test_idx in skf.split(df, y_strat):
    
    X_train = df.iloc[train_idx].drop(columns=['days_to_close'])
    X_test  = df.iloc[test_idx].drop(columns=['days_to_close'])
    
    y_train = df.iloc[train_idx]['days_to_close']
    y_test  = df.iloc[test_idx]['days_to_close']
    
    # preprocessing (fit ONLY on train fold)
    X_train_enc = preprocessor.fit_transform(X_train)
    X_test_enc  = preprocessor.transform(X_test)
    
    # model
    lr = LinearRegression()
    lr.fit(X_train_enc, y_train)
    
    y_pred = lr.predict(X_test_enc)
    
    mae_scores.append(mean_absolute_error(y_test, y_pred))
    r2_scores.append(r2_score(y_test, y_pred))

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories

In [24]:
from statistics import mean, stdev

print("MAE:")
print("  Mean:", round(mean(mae_scores), 3))
print("  Std: ", round(stdev(mae_scores), 3))

print("\nR2:")
print("  Mean:", round(mean(r2_scores), 3))
print("  Std: ", round(stdev(r2_scores), 3))

MAE:
  Mean: 1.86
  Std:  0.013

R2:
  Mean: 0.375
  Std:  0.007


### There are too many unique problems
Looking at my results from both the basic train/test split and from the stratfied k-fold cross validation, I think that I should ptobably drop the problem catagory from my feature set.

In [25]:

df['problem'].value_counts().head(60)

problem
Illegal Parking                   28835
HEAT/HOT WATER                    28230
Noise - Residential               16622
Snow or Ice                       11807
Blocked Driveway                  11033
Street Condition                   6380
Traffic Signal Condition           5046
Noise - Street/Sidewalk            3570
Water System                       3568
Dirty Condition                    3359
PLUMBING                           3279
UNSANITARY CONDITION               3162
Abandoned Vehicle                  2824
PAINT/PLASTER                      2381
Noise                              2327
Noise - Commercial                 2274
Noise - Vehicle                    1702
WATER LEAK                         1616
General Construction/Plumbing      1493
DOOR/WINDOW                        1476
Derelict Vehicles                  1338
Elevator                           1309
Damaged Tree                       1289
ELECTRIC                           1249
Illegal Dumping                 

### Redoing the basic model, but ignoring the problem column. 

In [26]:
y = df['days_to_close']
X = df.drop(columns=['days_to_close', 'problem'])
X

,agency,borough,incident_zip,day_of_week,hour_of_day,same_day_complaint_volume
0,DCWP,BRONX,10455.0,5,13,19
1,DCWP,BROOKLYN,11207.0,5,11,19
2,DCWP,BROOKLYN,11234.0,5,3,19
3,DCWP,BRONX,10454.0,5,11,19
4,DCWP,QUEENS,11423.0,5,15,19
...,...,...,...,...,...,...
173846,TLC,MANHATTAN,10028.0,1,16,4
173847,TLC,MANHATTAN,10017.0,1,9,4
173848,TLC,MANHATTAN,10036.0,1,12,4
173849,TLC,QUEENS,11430.0,4,15,1


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [28]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols),
    ]
)

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
feature_names = ohe_names + num_cols

In [29]:
lr = LinearRegression()
lr.fit(X_train_enc, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [30]:
y_pred = lr.predict(X_test_enc)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2:  {r2:.3f}")
print(f"MAE: {mae:.3f}")

R2:  0.189
MAE: 2.297


In [31]:
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': lr.coef_
}).sort_values(by='coefficient', key=abs, ascending=False)

coef_df.head(15)

,feature,coefficient
10,agency_OOS,25.056318
9,agency_NYPD,-9.541777
1,agency_DHS,-8.340213
0,agency_DEP,-7.865303
3,agency_DOE,-7.832412
5,agency_DOT,-7.621025
4,agency_DOHMH,-7.429877
7,agency_DSNY,-7.054118
11,agency_OTI,-6.171768
6,agency_DPR,-6.047059


Ok, so it seems like the specific problem might actually be a very significant feature, beacuse my R^2 got a lot worse once I removed the problem column entirely. I think that instead, what I should do is backwards selection. However, I think I should still filter out uncommon problems. 

[https://scikit-learn.org/0.24/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html](https://scikit-learn.org/0.24/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html)

In [32]:
problem_counts = df['problem'].value_counts()
rare_problems = problem_counts[problem_counts < 100].index

df['problem'] = df['problem'].replace(rare_problems, 'OTHER')

In [33]:
from sklearn.feature_selection import SequentialFeatureSelector
lr = LinearRegression()

sfs = SequentialFeatureSelector(
    lr,
    n_features_to_select=10,
    direction='backward',
    scoring='r2',
    cv=5,
    n_jobs=-1
)

sfs.fit(X_train_enc, y_train)

,estimator estimator: estimator instanceAn unfitted estimator.,LinearRegression()
,"n_features_to_select n_features_to_select: ""auto"", int or float, default=""auto""If `""auto""`, the behaviour depends on the `tol` parameter:- if `tol` is not `None`, then features are selected while the score change does not exceed `tol`.- otherwise, half of the features are selected.If integer, the parameter is the absolute number of features to select.If float between 0 and 1, it is the fraction of features to select... versionadded:: 1.1 The option `""auto""` was added in version 1.1... versionchanged:: 1.3 The default changed from `""warn""` to `""auto""` in 1.3.",10
,"tol tol: float, default=NoneIf the score is not incremented by at least `tol` between twoconsecutive feature additions or removals, stop adding or removing.`tol` can be negative when removing features using `direction=""backward""`.`tol` is required to be strictly positive when doing forward selection.It can be useful to reduce the number of features at the cost of a smalldecrease in the score.`tol` is enabled only when `n_features_to_select` is `""auto""`... versionadded:: 1.1",None
,"direction direction: {'forward', 'backward'}, default='forward'Whether to perform forward selection or backward selection.",'backward'
,"scoring scoring: str or callable, default=NoneScoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)`` that returns a single value. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion ` is used.",'r2'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. In all othercases, :class:`~sklearn.model_selection.KFold` is used. These splittersare instantiated with `shuffle=False` so the splits will be the sameacross calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel. When evaluating a new feature toadd or remove, the cross-validation procedure is parallel over thefolds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None


In [34]:
selected_mask = sfs.get_support()

selected_features = [
    feature_names[i]
    for i in range(len(feature_names))
    if selected_mask[i]
]

print("Selected features:")
for f in selected_features:
    print(f)

Selected features:
agency_DEP
agency_DHS
agency_DOB
agency_DOHMH
agency_DOT
agency_DPR
agency_DSNY
agency_NYPD
agency_OOS
same_day_complaint_volume


In [35]:
X_train_sel = X_train_enc[:, selected_mask]
X_test_sel = X_test_enc[:, selected_mask]

print("Selected train shape:", X_train_sel.shape)
print("Selected test shape: ", X_test_sel.shape)

Selected train shape: (137668, 10)
Selected test shape:  (34418, 10)


In [36]:
lr_selected = LinearRegression()
lr_selected.fit(X_train_sel, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [37]:
y_pred_sel = lr_selected.predict(X_test_sel)

print(f"Reduced-model R2:  {r2_score(y_test, y_pred_sel):.3f}")
print(f"Reduced-model MAE: {mean_absolute_error(y_test, y_pred_sel):.3f}")

Reduced-model R2:  0.184
Reduced-model MAE: 2.290


In [38]:
coef_df_sel = pd.DataFrame({
    'feature': selected_features,
    'coefficient': lr_selected.coef_
}).sort_values(by='coefficient', key=abs, ascending=False)

coef_df_sel

,feature,coefficient
8,agency_OOS,29.168464
7,agency_NYPD,-5.178823
1,agency_DHS,-4.113448
0,agency_DEP,-3.499629
4,agency_DOT,-3.259041
3,agency_DOHMH,-3.106520
6,agency_DSNY,-2.675628
5,agency_DPR,-1.685557
2,agency_DOB,-1.420748
9,same_day_complaint_volume,-0.775884
